In [1]:
import pandas as pd
import os
import cv2
import random
from pandas import read_excel
from pandas import DataFrame
from sklearn.decomposition import PCA
import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from xgboost import XGBRegressor
import catboost as cb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
import matplotlib.pyplot as plt
from functions import *
import time

In [3]:
file_name = 'data_train_augm_Pb.xlsx'
df = read_excel(file_name, sheet_name = 'Relavage 4')

ValueError: Worksheet named 'Relavage 4' not found

In [ ]:
test_file_name = 'data_test_augm_Pb.xlsx'
df_test = read_excel(test_file_name, sheet_name = 'Relavage 4')

In [ ]:
def image_flatten(image):
    # read the image
    img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    img = img.flatten()
    return img

In [ ]:
def image_to_fft_flatten(image):
    # read the image
    img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # apply 2d fft
    f = np.fft.fft2(img)
    # shift low frequency components to the center of the spectrum
    fshift = np.fft.fftshift(f)
    # fft results are complex numbers
    # so we need to compute the magnitude spectrum of the complex numbers (module in french)
    # mod(z) = |a + bi| = sqrt(a² + b²)
    magnitude_spectrum = np.log(np.sqrt(fshift.real**2 + fshift.imag**2)) # we apply log to make the spectrum more visible
    features = magnitude_spectrum.flatten()
    return features

Image flatten

In [ ]:
features = []
labels_cu = []
labels_pb = []
labels_fe = []
labels_zn = []
for line in df.values:
    image = cv2.imread("../train_resizedimages/"+line[0])
    feature = image_flatten(image)
    features.append(feature)
    labels_cu.append(line[1])
    labels_fe.append(line[2])
    labels_pb.append(line[3])
    labels_zn.append(line[4])

In [ ]:
features = np.array(features)
labels_cu = np.array(labels_cu)
labels_fe = np.array(labels_fe)
labels_pb = np.array(labels_pb)
labels_zn = np.array(labels_zn)

In [ ]:
labels = np.array([labels_cu, labels_fe, labels_pb, labels_zn]).T

In [ ]:
svm = SVR()
msvm = MultiOutputRegressor(svm)
Start_Time = time.time()
msvm.fit(features, labels)
svm_time_train = time.time() - Start_Time

In [10]:
lr = LinearRegression()
mlr = MultiOutputRegressor(lr)
Start_Time = time.time()
mlr.fit(features, labels)
lr_time_train = time.time() - Start_Time

In [ ]:
# xgb = XGBRegressor()
# mxgb = MultiOutputRegressor(xgb)
# Start_Time = time.time()
# mxgb.fit(features, labels)
# xgb_time_train = time.time() - Start_Time

In [11]:
xgb_time_train = "> 1h"

In [12]:
# cbr = cb.CatBoostRegressor()
# mcb = MultiOutputRegressor(cbr)
# Start_Time = time.time()
# mcb.fit(features, labels, verbose=False)
# cb_time_train = time.time() - Start_Time

In [13]:
cb_time_train = "> 1h"

In [14]:
# rf = RandomForestRegressor()
# mrf = MultiOutputRegressor(rf)
# Start_Time = time.time()
# mrf.fit(features, labels)
# rf_time_train = time.time() - Start_Time

In [15]:
rf_time_train = "> 1h"

test videos

In [16]:
test_features = []
test_labels_cu = []
test_labels_fe = []
test_labels_pb = []
test_labels_zn = []
for line in df_test.values:
    image = cv2.imread("../test_resizedimages/"+line[0])
    fft_pca = image_flatten(image)
    test_features.append(fft_pca)
    test_labels_cu.append(line[1])
    test_labels_fe.append(line[2])
    test_labels_pb.append(line[3])
    test_labels_zn.append(line[4])

In [17]:
test_features = np.array(test_features)
test_labels_cu = np.array(test_labels_cu)
test_labels_fe = np.array(test_labels_fe)
test_labels_pb = np.array(test_labels_pb)
test_labels_zn = np.array(test_labels_zn)

In [18]:
test_labels = np.array([test_labels_cu, test_labels_fe, test_labels_pb, test_labels_zn]).T

In [19]:
y_pred = msvm.predict(test_features)
svm_mae = mean_absolute_error(test_labels, y_pred)
svm_mse = mean_squared_error(test_labels, y_pred)
svm_r2 = r2_score(test_labels, y_pred)

In [20]:
# y_pred = mxgb.predict(test_features)
# xgb_mae = mean_absolute_error(test_labels, y_pred)
# xgb_mse = mean_squared_error(test_labels, y_pred)
# xgb_r2 = r2_score(test_labels, y_pred)

In [21]:
xgb_mae, xgb_mse, xgb_r2 = None, None, None

In [22]:
y_pred = mlr.predict(test_features)
lr_mae = mean_absolute_error(test_labels, y_pred)
lr_mse = mean_squared_error(test_labels, y_pred)
lr_r2 = r2_score(test_labels, y_pred)

In [23]:
# y_pred = mcb.predict(test_features)
# cb_mae = mean_absolute_error(test_labels, y_pred)
# cb_mse = mean_squared_error(test_labels, y_pred)
# cb_r2 = r2_score(test_labels, y_pred)

In [24]:
cb_mae, cb_mse, cb_r2 = None, None, None

In [ ]:
# y_pred = mrf.predict(test_features)
# rf_mae = mean_absolute_error(test_labels, y_pred)
# rf_mse = mean_squared_error(test_labels, y_pred)
# rf_r2 = r2_score(test_labels, y_pred)

In [25]:
rf_mae, rf_mse, rf_r2 = None, None, None

In [26]:
image_flatten = pd.DataFrame(columns=["Model", "MAE", "MSE", "R2 Score", "Time"])
image_flatten = image_flatten.append({"Model": "SVM", "MAE": svm_mae, "MSE": svm_mse, "R2 Score": svm_r2, "Time": svm_time_train}, ignore_index=True)
image_flatten = image_flatten.append({"Model": "XGBoost", "MAE": xgb_mae, "MSE": xgb_mse, "R2 Score": xgb_r2, "Time": xgb_time_train}, ignore_index=True)
image_flatten = image_flatten.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE": lr_mse, "R2 Score": lr_r2, "Time": lr_time_train}, ignore_index=True)
image_flatten = image_flatten.append({"Model": "CatBoost", "MAE": cb_mae, "MSE": cb_mse, "R2 Score": cb_r2, "Time": cb_time_train}, ignore_index=True)
image_flatten = image_flatten.append({"Model": "Random Forest", "MAE": rf_mae, "MSE": rf_mse, "R2 Score": rf_r2, "Time": rf_time_train}, ignore_index=True)
image_flatten

C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\2676219377.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  image_flatten = image_flatten.append({"Model": "SVM", "MAE": svm_mae, "MSE": svm_mse, "R2 Score": svm_r2, "Time": svm_time_train}, ignore_index=True)
C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\2676219377.py:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  image_flatten = image_flatten.append({"Model": "XGBoost", "MAE": xgb_mae, "MSE": xgb_mse, "R2 Score": xgb_r2, "Time": xgb_time_train}, ignore_index=True)
C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\2676219377.py:4: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  image_flatten = image_flatten.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE

,Model,MAE,MSE,R2 Score,Time
0,SVM,2.580855,10.721311,-2.104973,186.753827
1,XGBoost,None,None,None,> 1h
2,Linear Regression,3.066946,20.058075,-3.735954,113.357589
3,CatBoost,None,None,None,> 1h
4,Random Forest,None,None,None,> 1h


FFT flatten

In [27]:
features = []
labels_cu = []
labels_pb = []
labels_fe = []
labels_zn = []
for line in df.values:
    image = cv2.imread("../train_resizedimages/"+line[0])
    feature = image_to_fft_flatten(image)
    features.append(feature)
    labels_cu.append(line[1])
    labels_fe.append(line[2])
    labels_pb.append(line[3])
    labels_zn.append(line[4])

In [28]:
features = np.array(features)
labels_cu = np.array(labels_cu)
labels_fe = np.array(labels_fe)
labels_pb = np.array(labels_pb)
labels_zn = np.array(labels_zn)

In [29]:
labels = np.array([labels_cu, labels_fe, labels_pb, labels_zn]).T

In [ ]:
svm = SVR()
msvm = MultiOutputRegressor(svm)
Start_Time = time.time()
msvm.fit(features, labels)
svm_time_train = time.time() - Start_Time

In [ ]:
lr = LinearRegression()
mlr = MultiOutputRegressor(lr)
Start_Time = time.time()
mlr.fit(features, labels)
lr_time_train = time.time() - Start_Time

In [ ]:
xgb = XGBRegressor()
mxgb = MultiOutputRegressor(xgb)
Start_Time = time.time()
mxgb.fit(features, labels)
xgb_time_train = time.time() - Start_Time

In [ ]:
# cbr = cb.CatBoostRegressor()
# mcb = MultiOutputRegressor(cbr)
# Start_Time = time.time()
# mcb.fit(features, labels, verbose=False)
# cb_time_train = time.time() - Start_Time

In [ ]:
# rf = RandomForestRegressor()
# mrf = MultiOutputRegressor(rf)
# Start_Time = time.time()
# mrf.fit(features, labels)
# rf_time_train = time.time() - Start_Time

test videos

In [43]:
test_features = []
test_labels_cu = []
test_labels_fe = []
test_labels_pb = []
test_labels_zn = []
for line in df_test.values:
    image = cv2.imread("../test_resizedimages/"+line[0])
    fft_pca = image_to_fft_flatten(image)
    test_features.append(fft_pca)
    test_labels_cu.append(line[1])
    test_labels_fe.append(line[2])
    test_labels_pb.append(line[3])
    test_labels_zn.append(line[4])

In [44]:
test_features = np.array(test_features)
test_labels_cu = np.array(test_labels_cu)
test_labels_fe = np.array(test_labels_fe)
test_labels_pb = np.array(test_labels_pb)
test_labels_zn = np.array(test_labels_zn)

In [45]:
test_labels = np.array([test_labels_cu, test_labels_fe, test_labels_pb, test_labels_zn]).T

In [46]:
y_pred = msvm.predict(test_features)
svm_mae = mean_absolute_error(test_labels, y_pred)
svm_mse = mean_squared_error(test_labels, y_pred)
svm_r2 = r2_score(test_labels, y_pred)

In [47]:
y_pred = mxgb.predict(test_features)
xgb_mae = mean_absolute_error(test_labels, y_pred)
xgb_mse = mean_squared_error(test_labels, y_pred)
xgb_r2 = r2_score(test_labels, y_pred)

In [48]:
y_pred = mlr.predict(test_features)
lr_mae = mean_absolute_error(test_labels, y_pred)
lr_mse = mean_squared_error(test_labels, y_pred)
lr_r2 = r2_score(test_labels, y_pred)

In [ ]:
# y_pred = mcb.predict(test_features)
# cb_mae = mean_absolute_error(test_labels, y_pred)
# cb_mse = mean_squared_error(test_labels, y_pred)
# cb_r2 = r2_score(test_labels, y_pred)

In [ ]:
# y_pred = mrf.predict(test_features)
# rf_mae = mean_absolute_error(test_labels, y_pred)
# rf_mse = mean_squared_error(test_labels, y_pred)
# rf_r2 = r2_score(test_labels, y_pred)

In [49]:
fft_flatten = pd.DataFrame(columns=["Model", "MAE", "MSE", "R2 Score", "Time"])
fft_flatten = fft_flatten.append({"Model": "SVM", "MAE": svm_mae, "MSE": svm_mse, "R2 Score": svm_r2, "Time": svm_time_train}, ignore_index=True)
fft_flatten = fft_flatten.append({"Model": "XGBoost", "MAE": xgb_mae, "MSE": xgb_mse, "R2 Score": xgb_r2, "Time": xgb_time_train}, ignore_index=True)
fft_flatten = fft_flatten.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE": lr_mse, "R2 Score": lr_r2, "Time": lr_time_train}, ignore_index=True)
fft_flatten = fft_flatten.append({"Model": "CatBoost", "MAE": cb_mae, "MSE": cb_mse, "R2 Score": cb_r2, "Time": cb_time_train}, ignore_index=True)
fft_flatten = fft_flatten.append({"Model": "Random Forest", "MAE": rf_mae, "MSE": rf_mse, "R2 Score": rf_r2, "Time": rf_time_train}, ignore_index=True)
fft_flatten

C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\11331139.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  fft_flatten = fft_flatten.append({"Model": "SVM", "MAE": svm_mae, "MSE": svm_mse, "R2 Score": svm_r2, "Time": svm_time_train}, ignore_index=True)
C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\11331139.py:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  fft_flatten = fft_flatten.append({"Model": "XGBoost", "MAE": xgb_mae, "MSE": xgb_mse, "R2 Score": xgb_r2, "Time": xgb_time_train}, ignore_index=True)
C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\11331139.py:4: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  fft_flatten = fft_flatten.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE": lr_mse, "R2 Sco

,Model,MAE,MSE,R2 Score,Time
0,SVM,2.290585,7.901094,-2.227423,179.789047
1,XGBoost,2.849284,15.56437,-2.167488,2079.502756
2,Linear Regression,2.348545,9.718011,-1.488427,110.186407
3,CatBoost,None,None,None,> 1h
4,Random Forest,None,None,None,> 1h


FFT PCA

In [50]:
features = []
labels_cu = []
labels_pb = []
labels_fe = []
labels_zn = []
for line in df.values:
    image = cv2.imread("../train_resizedimages/"+line[0])
    feature = image_to_fft_pca(image)
    features.append(feature)
    labels_cu.append(line[1])
    labels_fe.append(line[2])
    labels_pb.append(line[3])
    labels_zn.append(line[4])

In [51]:
features = np.array(features)
labels_cu = np.array(labels_cu)
labels_fe = np.array(labels_fe)
labels_pb = np.array(labels_pb)
labels_zn = np.array(labels_zn)

In [52]:
labels = np.array([labels_cu, labels_fe, labels_pb, labels_zn]).T

In [53]:
svm = SVR()
msvm = MultiOutputRegressor(svm)
Start_Time = time.time()
msvm.fit(features, labels)
svm_time_train = time.time() - Start_Time

In [54]:
lr = LinearRegression()
mlr = MultiOutputRegressor(lr)
Start_Time = time.time()
mlr.fit(features, labels)
lr_time_train = time.time() - Start_Time

In [55]:
xgb = XGBRegressor()
mxgb = MultiOutputRegressor(xgb)
Start_Time = time.time()
mxgb.fit(features, labels)
xgb_time_train = time.time() - Start_Time

In [56]:
cbr = cb.CatBoostRegressor()
mcb = MultiOutputRegressor(cbr)
Start_Time = time.time()
mcb.fit(features, labels, verbose=False)
cb_time_train = time.time() - Start_Time

In [57]:
rf = RandomForestRegressor()
mrf = MultiOutputRegressor(rf)
Start_Time = time.time()
mrf.fit(features, labels)
rf_time_train = time.time() - Start_Time

test videos

In [58]:
test_features = []
test_labels_cu = []
test_labels_fe = []
test_labels_pb = []
test_labels_zn = []
for line in df_test.values:
    image = cv2.imread("../test_resizedimages/"+line[0])
    fft_pca = image_to_fft_pca(image)
    test_features.append(fft_pca)
    test_labels_cu.append(line[1])
    test_labels_fe.append(line[2])
    test_labels_pb.append(line[3])
    test_labels_zn.append(line[4])

In [59]:
test_features = np.array(test_features)
test_labels_cu = np.array(test_labels_cu)
test_labels_fe = np.array(test_labels_fe)
test_labels_pb = np.array(test_labels_pb)
test_labels_zn = np.array(test_labels_zn)

In [60]:
test_labels = np.array([test_labels_cu, test_labels_fe, test_labels_pb, test_labels_zn]).T

In [61]:
y_pred = msvm.predict(test_features)
svm_mae = mean_absolute_error(test_labels, y_pred)
svm_mse = mean_squared_error(test_labels, y_pred)
svm_r2 = r2_score(test_labels, y_pred)

In [62]:
y_pred = mxgb.predict(test_features)
xgb_mae = mean_absolute_error(test_labels, y_pred)
xgb_mse = mean_squared_error(test_labels, y_pred)
xgb_r2 = r2_score(test_labels, y_pred)

In [63]:
y_pred = mlr.predict(test_features)
lr_mae = mean_absolute_error(test_labels, y_pred)
lr_mse = mean_squared_error(test_labels, y_pred)
lr_r2 = r2_score(test_labels, y_pred)

In [64]:
y_pred = mcb.predict(test_features)
cb_mae = mean_absolute_error(test_labels, y_pred)
cb_mse = mean_squared_error(test_labels, y_pred)
cb_r2 = r2_score(test_labels, y_pred)

In [65]:
y_pred = mrf.predict(test_features)
rf_mae = mean_absolute_error(test_labels, y_pred)
rf_mse = mean_squared_error(test_labels, y_pred)
rf_r2 = r2_score(test_labels, y_pred)

In [66]:
fft_pca = pd.DataFrame(columns=["Model", "MAE", "MSE", "R2 Score", "Time"])
fft_pca = fft_pca.append({"Model": "SVM", "MAE": svm_mae, "MSE": svm_mse, "R2 Score": svm_r2, "Time": svm_time_train}, ignore_index=True)
fft_pca = fft_pca.append({"Model": "XGBoost", "MAE": xgb_mae, "MSE": xgb_mse, "R2 Score": xgb_r2, "Time": xgb_time_train}, ignore_index=True)
fft_pca = fft_pca.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE": lr_mse, "R2 Score": lr_r2, "Time": lr_time_train}, ignore_index=True)
fft_pca = fft_pca.append({"Model": "CatBoost", "MAE": cb_mae, "MSE": cb_mse, "R2 Score": cb_r2, "Time": cb_time_train}, ignore_index=True)
fft_pca = fft_pca.append({"Model": "Random Forest", "MAE": rf_mae, "MSE": rf_mse, "R2 Score": rf_r2, "Time": rf_time_train}, ignore_index=True)
fft_pca

C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\1515366811.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  fft_pca = fft_pca.append({"Model": "SVM", "MAE": svm_mae, "MSE": svm_mse, "R2 Score": svm_r2, "Time": svm_time_train}, ignore_index=True)
C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\1515366811.py:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  fft_pca = fft_pca.append({"Model": "XGBoost", "MAE": xgb_mae, "MSE": xgb_mse, "R2 Score": xgb_r2, "Time": xgb_time_train}, ignore_index=True)
C:\Users\Youssef\AppData\Local\Temp\ipykernel_7916\1515366811.py:4: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  fft_pca = fft_pca.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE": lr_mse, "R2 Score": lr_r2, "Time"

,Model,MAE,MSE,R2 Score,Time
0,SVM,2.430637,9.796657,-2.371885,0.386961
1,XGBoost,2.935539,16.070653,-2.574089,9.412806
2,Linear Regression,3.069723,22.574882,-3.053115,0.601674
3,CatBoost,2.803575,13.005596,-2.129712,314.104261
4,Random Forest,2.705838,11.494991,-1.882444,81.658593
